# 07 — The Retrieval Pipeline: Wiring It All Together

Everything so far has been a separate piece: chunking (notebook 06), embeddings (notebook 04), a vector store (notebook 05). Today we bolt them into one pipeline that takes a folder of raw story files and a question, and hands back the most relevant chunks.

```
Raw story files
     |
     v
[ Load ]
     |
     v
[ Chunk ]
     |
     v
[ Embed ]
     |
     v
[ Store in a vector index ]        <- happens once, at index time
─────────────────────────────
User question                       <- happens on every query
     |
     v
[ Embed the question ]
     |
     v
[ Similarity search ]
     |
     v
Retrieved chunks
```


In [ ]:
import os
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter

embedder = SentenceTransformer("all-MiniLM-L6-v2")

DATA_DIR = os.path.join("..", "data")
STORIES_DIR = os.path.join(DATA_DIR, "stories")


def load_documents(folder_path: str) -> list[dict]:
    docs = []
    for filename in sorted(os.listdir(folder_path)):
        if filename.endswith(".txt"):
            with open(os.path.join(folder_path, filename)) as f:
                docs.append({"filename": filename, "content": f.read()})
    return docs


## Phase 1 — indexing (runs once)


In [ ]:
def index_documents(folder_path: str, chunk_size: int = 300, chunk_overlap: int = 50):
    """Load, chunk, embed, and index every document in folder_path.

    Returns (faiss_index, doc_store) -- doc_store[i] holds the metadata
    for the vector at row i of the FAISS index.
    """
    splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    docs = load_documents(folder_path)

    doc_store = []
    for doc in docs:
        chunks = splitter.split_text(doc["content"])
        for i, chunk in enumerate(chunks):
            doc_store.append({
                "content": chunk,
                "metadata": {"source": doc["filename"], "chunk_index": i},
            })

    chunk_embeddings = embedder.encode([d["content"] for d in doc_store]).astype(np.float32)
    faiss.normalize_L2(chunk_embeddings)

    index = faiss.IndexFlatIP(chunk_embeddings.shape[1])
    index.add(chunk_embeddings)

    return index, doc_store


index, doc_store = index_documents(STORIES_DIR)
print(f"Indexed {len(doc_store)} chunks from {len(load_documents(STORIES_DIR))} files")
print(f"Embedding dimension: {index.d}")


This runs **once**, whenever your documents change — not on every question. Keep that distinction in mind; it's the difference between an app that's fast and one that re-embeds your whole library on every keystroke.


## Phase 2 — retrieval (runs on every query)


In [ ]:
def retrieve(query: str, index, doc_store, top_k: int = 3) -> list[dict]:
    query_vec = embedder.encode([query]).astype(np.float32)
    faiss.normalize_L2(query_vec)
    scores, indices = index.search(query_vec, top_k)
    return [
        {**doc_store[i], "score": float(scores[0][j])}
        for j, i in enumerate(indices[0])
    ]


results = retrieve("what happens in the wormhole", index, doc_store, top_k=3)
for r in results:
    print(f"Rank (score {r['score']:.2f}) | {r['metadata']['source']} chunk #{r['metadata']['chunk_index']}")
    print(f"  {r['content']!r}")
    print()


## When retrieval should say "nothing relevant"

FAISS always returns your `top_k` best matches — even for a question that has nothing to do with any of your documents. Watch what happens with something totally unrelated to StoryVerse.


In [ ]:
irrelevant_results = retrieve("what is the best Python ORM for a Django project?", index, doc_store, top_k=3)
for r in irrelevant_results:
    print(f"(score {r['score']:.2f}) {r['metadata']['source']}")


The scores here should be noticeably lower than the wormhole query above — but they're never zero, because "best match out of what we have" is all similarity search can promise. We need a cutoff.


In [ ]:
def retrieve_with_threshold(query: str, index, doc_store, top_k: int = 3, min_score: float = 0.35) -> list[dict]:
    results = retrieve(query, index, doc_store, top_k)
    return [r for r in results if r["score"] >= min_score]


filtered = retrieve_with_threshold("what is the best Python ORM for a Django project?", index, doc_store, min_score=0.35)
print(f"Results above threshold: {len(filtered)}")


Without a score threshold, an irrelevant question still hands the model *something* to work with — and a model asked to answer from low-relevance context will often try anyway, which is exactly how hallucination sneaks back in even after you've built retrieval. A threshold is your first real defense against that.


## The same pipeline, the LangChain way


In [ ]:
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings

lc_embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

lc_vectorstore = Chroma.from_texts(
    texts=[d["content"] for d in doc_store],
    embedding=lc_embeddings,
    metadatas=[d["metadata"] for d in doc_store],
    collection_name="storyverse_chunks",
)

retriever = lc_vectorstore.as_retriever(search_kwargs={"k": 3})
lc_results = retriever.invoke("what happens in the wormhole")

for doc in lc_results:
    print(doc.metadata, "->", doc.page_content[:80], "...")


Same chunks come back, wrapped as LangChain `Document` objects instead of our own dicts. `.as_retriever()` standardizes the "given a query, return relevant chunks" interface so you can swap the underlying vector store (FAISS, Chroma, Pinecone, whatever) without touching the code that calls it.


## Checking retrieval quality by hand

Two questions per story, ten total — read each result and judge for yourself whether the top chunk is actually relevant.


In [ ]:
quality_check_queries = [
    "Who is Cooper's daughter?",
    "What happens inside the tesseract?",
    "Who are Harry's two best friends?",
    "What object is Voldemort trying to steal?",
    "Who does Shivudu fall in love with?",
    "Who is the villain that betrayed Shivudu's father?",
    "What does Arjun find behind the bookshop's second door?",
    "How many moons are in the sky Arjun finds himself under?",
    "Why does Batman take the blame for Harvey Dent's crimes?",
    "Who is the criminal mastermind causing chaos in Gotham?",
]

for q in quality_check_queries:
    top = retrieve(q, index, doc_store, top_k=1)[0]
    print(f"Q: {q}")
    print(f"   -> ({top['score']:.2f}) {top['metadata']['source']}: {top['content'][:90]}...")
    print()


Somewhere in that list you'll likely find at least one borderline or wrong result — maybe a question whose answer spans two chunks, or one that's ambiguous across two stories. That's normal. Perfect retrieval is hard, and notebook 09 is dedicated entirely to fixing cases like that. For now, just notice which ones felt shaky.


## Persisting the index

Indexing takes real time once your document count grows. Always save it — only re-index when documents actually change.


In [ ]:
import json

faiss_chunks_path = os.path.join(DATA_DIR, "storyverse_chunks.faiss")
doc_store_path = os.path.join(DATA_DIR, "storyverse_chunks_docstore.json")

faiss.write_index(index, faiss_chunks_path)
with open(doc_store_path, "w") as f:
    json.dump(doc_store, f)

print(f"Saved index to {faiss_chunks_path}")
print(f"Saved doc store to {doc_store_path}")

# Reload, to prove it round-trips
reloaded_index = faiss.read_index(faiss_chunks_path)
with open(doc_store_path) as f:
    reloaded_doc_store = json.load(f)

print(f"Reloaded {reloaded_index.ntotal} vectors and {len(reloaded_doc_store)} doc_store entries")


In [ ]:
chroma_persist_dir = os.path.join(DATA_DIR, "chroma_db")
persistent_client_vectorstore = Chroma.from_texts(
    texts=[d["content"] for d in doc_store],
    embedding=lc_embeddings,
    metadatas=[d["metadata"] for d in doc_store],
    collection_name="storyverse_chunks",
    persist_directory=chroma_persist_dir,
)
print(f"Chroma collection persisted to {chroma_persist_dir}")


## What we have now

```
Stories -> Chunks -> Embeddings -> FAISS index      [build once]
Question -> Embed -> Search -> Top-K chunks          [per query]
```

We can reliably turn a folder of documents into a searchable index, and a question into the right handful of chunks. The one piece still missing: we retrieve chunks, but we don't yet hand them to a model to actually *answer* the question. That's the very next notebook.

**Terms to keep, in plain English:**

| Term | Plain meaning |
|---|---|
| Indexing | The one-time step of chunking + embedding + storing documents |
| Retrieval | The per-query step of finding the most relevant stored chunks |
| Score threshold | A minimum similarity score below which results get discarded |
| Retriever | A standard "query in, relevant documents out" interface |

**Exercises:**

- Try `top_k` values of 1, 3, 5, and 10 on the same query — how does the retrieved context change?
- Try `min_score` values of 0.2, 0.35, and 0.5 — which queries get filtered out entirely at each threshold?
- Add a 6th story file to `STORIES_DIR`, re-run `index_documents`, and confirm it's retrievable.
